## Task 2 로직 테스트

- 1. 질문 키워드 분석 (조건들을 세부적으로 나누기)
- 2. yfinance에서 위 키워드를 바탕으로 어떤 값들을 가져와야 되는지 판단
- 3. 질문에 맞게 값 계산
- 4. 응답

In [1]:
import os
import dotenv
from langchain_naver import ChatClovaX

dotenv.load_dotenv()

chat = ChatClovaX(
    api_key=os.getenv('NCP_CLOVASTUDIO_API_KEY'),
    model="HCX-005",
    temperature=0.2,
    max_tokens=None,
    timeout=None,
    # max_retries,
    # repeat_penalt,
    # repetition_penalty,
    # stop,
    # seed,
    # top_k,
    # top_p,
)

In [82]:
prompt = """
# ROLE: system
당신은 주식 조건검색 문장을 구조화된 JSON 으로 변환하는 분석 전문가입니다.
다른 말은 절대 출력하지 말고, 아래 "반드시 지켜야 하는 스키마"에 100 % 맞는 JSON 만 출력하세요.

<스키마>
{{
  "type":           string,   # ↓ 표에서 하나만
  "date":           "YYYY-MM-DD",
  "market":         "KOSPI" | "KOSDAQ" | "ALL",
  "conditions": [
    {
      "field":      "change_rate" | "volume_change_rate" | "volume_abs" | "price",
      "operator":   ">" | ">=" | "<" | "<=" | "range",      
      "value":      number | None,   # ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓
      "min_value":  number | None,   # 모든 조건 객체에 반드시 포함, 없으면 None
      "max_value":  number | None,   # (키를 절대 생략하지 말 것!)
      "unit":       "percent" | "shares" | "won",
      "description": string
    }
  ],
  "keywords":   [string, ...],      # 최대 5개, 핵심 명사·숫자 위주
  "complexity": "simple" | "complex",
  "confidence": number,             # 0.0-1.0
  "reasoning":  string              # 30자 이하 요약
}}

<type 분류 표>
| 문장 특징                                    | type 값                    |
|----------------------------------------------|----------------------------|
| “KOSPI 시장 … 등락률 …”                      | KOSPI_등락률               |
| “KOSPI 시장 … 종가(=가격) …”                 | KOSPI_가격범위             |
| “KOSDAQ 시장 … 거래량 전날대비 …”            | KOSDAQ_거래량변화          |
| “KOSDAQ 시장 … 거래량 …주 이상”              | KOSDAQ_거래량              |
| “KOSDAQ 시장 … 종가 …원 이상/이하”           | KOSDAQ_가격범위            |
| “등락률 + 거래량변화” 같은 다중 조건 포함     | 복합조건                   |
| 시장 지정 없고 “등락률 …”만                  | 등락률                     |
| 시장 지정 없고 “거래량 …주 이상/이하”        | 거래량절대값               |
| 시장 지정 없고 “거래량 전날대비 …% 이상/이하”| 거래량변화율               |
| 시장 지정 없고 “종가 …원 이상/이하”          | 가격범위                   |

<필드 결정 규칙>
- “등락률” → field == change_rate, unit == percent
- “거래량이 전날대비” → field == volume_change_rate, unit == percent
- “거래량이 100만주” → field == volume_abs, unit == shares (숫자는 ‘만’, ‘천’ 단위 환산 하지 말고 실제 주식 수로 변환: 100만주 → 1000000)
- “종가가 …원” → field == price, unit == won (만원·천원·백원 단위도 숫자로 환산: 1만원 → 10000)

<연산자 규칙>
- “~이상” ⇒ operator == ">="
- “~초과” ⇒ ">"
- “~이하” ⇒ "<="
- “~미만” ⇒ "<"
- “A 이상 B 이하 범위” ⇒ operator == "range", min_value == A, max_value == B

<복잡도 판단>
- 조건 1개 ⇒ simple ; 2개 이상 ⇒ complex 

<출력 예시1 - 복합조건>
{{
  "type": "복합조건",
  "date": "2025-04-02",
  "market": "ALL",
  "conditions": [
    {
      "field": "change_rate",
      "operator": ">=",
      "value": 3,
      "unit": "percent",
      "min_value": None,
      "max_value": None,
      "description": "등락률 +3% 이상"
    },
    {
      "field": "volume_change_rate",
      "operator": ">=",
      "value": 100,
      "unit": "percent",
      "min_value": None,
      "max_value": None,
      "description": "거래량 전날대비 100% 이상 증가"
    }
  ],
  "keywords": ["2025-04-02", "등락률", "거래량", "3", "100"],
  "complexity": "complex",
  "confidence": 0.93,
  "reasoning": "두 조건 모두 전일 비교"
}}

<출력 예시 - 단일 조건>
{{
  "type": "거래량절대값",
  "date": "2024-11-22",
  "market": "ALL",
  "conditions": [
    {
      "field": "volume_abs",
      "operator": ">=",
      "value": 20000000,
      "min_value": None,
      "max_value": None,
      "unit": "shares",
      "description": "거래량이 2000만주 이상"
    }
  ],
  "keywords": ["2024-11-22", "거래량", "2000만주"],
  "complexity": "simple",
  "confidence": 0.95,
  "reasoning": "단일 조건으로 필터"
}}

"""

text = '2025-01-16에 등락률이 +7% 이상이면서 거래량이 전날대비 100% 이상 증가한 종목을 모두 보여줘'

messages = [("system", prompt),
            ("human", text)]

ai_msg = chat.invoke(messages)
ai_msg.content

'```json\n{\n  "type": "복합조건",\n  "date": "2025-01-16",\n  "market": "ALL",\n  "conditions": [\n    {\n      "field": "change_rate",\n      "operator": ">=",\n      "value": 7,\n      "unit": "percent",\n      "min_value": None,\n      "max_value": None,\n      "description": "등락률 +7% 이상"\n    },\n    {\n      "field": "volume_change_rate",\n      "operator": ">=",\n      "value": 100,\n      "unit": "percent",\n      "min_value": None,\n      "max_value": None,\n      "description": "거래량 전날대비 100% 이상 증가"\n    }\n  ],\n  "keywords": ["2025-01-16", "등락률", "거래량", "+7%", "100% 증가"],\n  "complexity": "complex",\n  "confidence": 0.9,\n  "reasoning": "두 가지 조건을 동시에 만족해야 함"\n}\n```'

In [83]:
def parse_llm_json(text: str) -> dict:
    import json, re, ast

    # 1)  {...} 블록만 추출
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        raise ValueError("JSON 블록을 찾을 수 없습니다.")
    j = m.group()

    # 2) JSON → dict
    try:
        return json.loads(j)
    except json.JSONDecodeError:
        return ast.literal_eval(j)   # fallback

parse_llm_json(ai_msg.content)

{'type': '복합조건',
 'date': '2025-01-16',
 'market': 'ALL',
 'conditions': [{'field': 'change_rate',
   'operator': '>=',
   'value': 7,
   'unit': 'percent',
   'min_value': None,
   'max_value': None,
   'description': '등락률 +7% 이상'},
  {'field': 'volume_change_rate',
   'operator': '>=',
   'value': 100,
   'unit': 'percent',
   'min_value': None,
   'max_value': None,
   'description': '거래량 전날대비 100% 이상 증가'}],
 'keywords': ['2025-01-16', '등락률', '거래량', '+7%', '100% 증가'],
 'complexity': 'complex',
 'confidence': 0.9,
 'reasoning': '두 가지 조건을 동시에 만족해야 함'}

In [84]:
import pandas as pd

llm_question_analysis = parse_llm_json(ai_msg.content)
market_list = pd.read_csv("/Users/yoon/MIRAE_ASSET_AI-Festival/krx_tickers_cleaned.csv")

def get_tickers(answer):
    if answer['market'] == "KOSPI":
        ticker_lists = market_list.loc[market_list["market"] == "KOSPI", "code"].tolist()
        # .KS 접미사 추가
        ticker_lists = [f"{ticker}.KS" for ticker in ticker_lists]
        print(len(ticker_lists))
        return ticker_lists
    elif answer['market'] == "KOSDAQ":
        ticker_lists = market_list.loc[market_list["market"] == "KOSDAQ", "code"].tolist()
        # .KQ 접미사 추가
        ticker_lists = [f"{ticker}.KQ" for ticker in ticker_lists]
        print(len(ticker_lists))
        return ticker_lists
    else:
        # 전체 시장인 경우 각각에 맞는 접미사 추가
        kospi_tickers = market_list.loc[market_list["market"] == "KOSPI", "code"].tolist()
        kosdaq_tickers = market_list.loc[market_list["market"] == "KOSDAQ", "code"].tolist()
        
        ticker_lists = [f"{ticker}.KS" for ticker in kospi_tickers] + \
                      [f"{ticker}.KQ" for ticker in kosdaq_tickers]
        print(len(ticker_lists))
        return ticker_lists

target_market = get_tickers(llm_question_analysis)

2757


In [85]:
import yfinance as yf
from typing import List
from datetime import datetime, timedelta
from typing import Dict, Callable, List, Set, Tuple
import numpy as np
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# 1. 유틸: 연산자 함수 테이블
OPS: Dict[str, Callable] = {
    ">":  lambda a, b: a > b,
    ">=": lambda a, b: a >= b,
    "<":  lambda a, b: a < b,
    "<=": lambda a, b: a <= b,
    "range": lambda a, min_val, max_val: min_val <= a <= max_val
}

def apply_operator(df, operator, value, min_value=None, max_value=None):
    """
    데이터프레임에 operator 조건을 적용하는 함수
    """
    if operator == "range":
        return (df >= min_value) & (df <= max_value)
    else:
        return OPS[operator](df, value)

# 2. 유틸: yfinance 에서 필요한 칼럼
FIELD_COLS = {
    "change_rate":        "Close",
    "volume_change_rate": "Volume",
    "volume_abs":         "Volume",
    "price":              "Close",
}

DATE_RANGE = {
    "change_rate":        2,
    "volume_change_rate": 2,
    "volume_abs":         1,
    "price":              1,
}

# 3. 전일 대비 지표 계산
def compute_metrics(df: pd.DataFrame, field: str) -> float:
    """
    한 종목의 가격·거래량 DataFrame → 지표 값(전일 대비 %, 절댓값 등)
    df.index: DatetimeIndex, 오름차순.
    """
    today = df.iloc[-1]
    prev  = df.iloc[-2] if len(df) >= 2 else None   # 전일이 없으면 None

    if field == "change_rate":
        if prev is None or prev["Close"] == 0:
            return float("nan")
        return (today["Close"] / prev["Close"] - 1) * 100

    if field == "volume_change_rate":
        if prev is None or prev["Volume"] == 0:
            return float("nan")
        return (today["Volume"] / prev["Volume"] - 1) * 100

    if field == "volume_abs":
        return today["Volume"]

    if field == "price":
        return today["Close"]

    raise ValueError(f"지원하지 않는 field: {field}")

In [86]:
import time
def download_in_batches(tickers, start_dt, end_dt, batch_size=100):
    """배치 단위로 다운로드 후 멀티인덱스 DataFrame으로 합치기"""
    all_ticker_data = {}
    total_batches = len(tickers) // batch_size + (1 if len(tickers) % batch_size > 0 else 0)
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        batch_num = i // batch_size + 1
        
        print(f"Processing batch {batch_num}/{total_batches} ({len(batch)} tickers)")
        
        try:
            # 배치 단위로 다운로드
            hist = yf.download(
                tickers=" ".join(batch),
                start=start_dt,
                end=end_dt,
                interval="1d",
                group_by="ticker",
                progress=False,
                threads=True,
                auto_adjust=False,
            )
            
            if not hist.empty:
                # 단일 종목인 경우 멀티인덱스로 변환
                if len(batch) == 1:
                    ticker = batch[0]
                    hist.columns = pd.MultiIndex.from_product([[ticker], hist.columns])
                
                # 각 ticker별로 데이터 저장
                for ticker in batch:
                    try:
                        if len(batch) == 1:
                            ticker_data = hist[ticker]
                        else:
                            ticker_data = hist[ticker]
                        
                        if not ticker_data.empty and not ticker_data.isna().all().all():
                            all_ticker_data[ticker] = ticker_data
                    except (KeyError, AttributeError):
                        print(f"No data for {ticker}")
                
                print(f"✓ Batch {batch_num} completed")
            else:
                print(f"✗ Batch {batch_num} returned empty data")
                
        except Exception as e:
            print(f"✗ Batch {batch_num} failed: {e}")
            # 실패한 배치는 개별 처리
            for ticker in batch:
                try:
                    single_hist = yf.download(ticker, start=start_dt, end=end_dt, progress=False)
                    if not single_hist.empty:
                        all_ticker_data[ticker] = single_hist
                        print(f"Individual download success: {ticker}")
                except:
                    print(f"Failed to download {ticker}")
        
        time.sleep(1)
    
    # 모든 ticker 데이터를 멀티인덱스 DataFrame으로 합치기
    if all_ticker_data:
        print(f"Combining data for {len(all_ticker_data)} tickers...")
        
        # 멀티인덱스 DataFrame 생성
        combined_data = pd.concat(all_ticker_data, axis=1)
        
        # 컬럼 순서 정리 (Price 기준으로)
        combined_data.columns.names = ['Ticker', 'Price']
        
        print(f"Final data shape: {combined_data.shape}")
        print(f"Columns: {combined_data.columns.get_level_values(1).unique().tolist()}")
        
        return combined_data
    else:
        print("No data downloaded")
        return pd.DataFrame()

In [87]:
def screen_stocks(q: dict) -> List[str]:
    """
    q : LLM JSON 파싱 결과.
    반환 : 모든 조건을 동시에 만족하는 종목 코드 리스트(중복 제거·정렬).
    """

    # 조사할 시장
    tickers  = get_tickers(q)
    if not tickers:
        return []
    
    # 종목 정보 로드
    stock_info = pd.read_csv('/Users/yoon/MIRAE_ASSET_AI-Festival/krx_tickers_cleaned.csv')  
    ticker_to_name = dict(zip(stock_info['code'], stock_info['name']))

    # 조건 수
    num_conditions = len(q['conditions'])
    print(f"조건 수 = 반복문 횟수 : {num_conditions}")

    condition_results: List[List[str]] = []

    stock_names = {}
    # 조건 수 만큼 개별적으로 돌기
    for i in range(num_conditions):

        print(f"{i+1}번째 반복문 돌아가는 중----------------------")

        date_range = DATE_RANGE[q['conditions'][i]['field']]
        print(f"date_range : {date_range}")

        # 거래량 / 종가 계산 컬럼 가져오기
        need_col = FIELD_COLS[q['conditions'][i]['field']]
        print(f"필요한 컬럼 : {need_col}")

        # 질문에 있는 날짜
        tgt_date  = datetime.strptime(q["date"], "%Y-%m-%d")
        print(f"타킷 날짜 : {tgt_date}")

        # 주말·휴일 여유
        start_dt  = tgt_date - timedelta(days=5)            
        print(f"조회 시작 날짜 : {start_dt}")

        # 끝 날짜
        end_dt    = tgt_date + timedelta(days=1)          
        print(f"조희 끝 날짜 : {end_dt}")

        # 배치 다운로드
        data = download_in_batches(tickers, start_dt, end_dt, batch_size=200)

        # 거래량 / 종가 계산 컬럼만 있는 데이터프레임
        only_need_col = data.xs(need_col, level=1, axis=1)

        # 전날 대비 거래량 / 등락률 계산
        if date_range == 2:
            only_need_col_edit = only_need_col.pct_change() * 100
            final_data = only_need_col_edit.iloc[-1]
            final_data = final_data.dropna()
            print("date_range = 2일 때 계산---------------------------")
            print(final_data.head(10))
        else:
            final_data = only_need_col.iloc[-1]
            final_data = final_data.dropna()
            print("date_range = 1일 때 계산---------------------------")
            print(final_data.head(10))

        # 연산자에 따른 계산 다르게
        def apply_operator(df, operator, value, min_value=None, max_value=None):
            """
            데이터프레임에 operator 조건을 적용하는 함수
            """
            if operator == "range":
                return (df >= min_value) & (df <= max_value)
            else:
                return OPS[operator](df, value)
            
        result = apply_operator(final_data, q['conditions'][i]['operator'], q['conditions'][i]['value'], q['conditions'][i]['min_value'], q['conditions'][i]['max_value'])

        # 조건 만족해서 True인 행의 Ticker들
        true_tickers = result[result == True].index.tolist()
        print("조건을 만족하는 ticker들 수:", len(true_tickers))
        # 조건별 결과 저장
        condition_results.append(true_tickers)

    # ---------- 교집합 계산 ----------
    if not condition_results:          # 보호 코드
        return {}

    # 조건이 1개면 그대로, 2개 이상이면 set 교집합
    if len(condition_results) == 1:
        final_tickers = condition_results[0]
    else:
        # map(set, …) 으로 각 리스트를 집합으로 변환 후 reduce 로 교차
        from functools import reduce
        final_tickers = list(
            reduce(set.intersection, map(set, condition_results)))

    print(f"최종 선별된 종목 수: {len(final_tickers)}")

    # 최종 결과를 딕셔너리로 변환
    stock_names = {}
    for ticker in final_tickers:
        clean_ticker = ticker.replace('.KS', '').replace('.KQ', '')
        stock_name = ticker_to_name.get(clean_ticker, ticker)
        stock_names[ticker] = stock_name

    return stock_names

In [88]:
screen_stocks(llm_question_analysis)

2757
조건 수 = 반복문 횟수 : 2
1번째 반복문 돌아가는 중----------------------
date_range : 2
필요한 컬럼 : Close
타킷 날짜 : 2025-01-16 00:00:00
조회 시작 날짜 : 2025-01-11 00:00:00
조희 끝 날짜 : 2025-01-17 00:00:00
Processing batch 1/14 (200 tickers)



1 Failed download:
['064400.KS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-01-11 00:00:00 -> 2025-01-17 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1736521200, endDate = 1737039600")')


✓ Batch 1 completed
Processing batch 2/14 (200 tickers)



1 Failed download:
['483650.KS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-01-11 00:00:00 -> 2025-01-17 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1736521200, endDate = 1737039600")')


✓ Batch 2 completed
Processing batch 3/14 (200 tickers)



2 Failed downloads:
['031210.KS', '480370.KS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-01-11 00:00:00 -> 2025-01-17 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1736521200, endDate = 1737039600")')


... (44 outputs left out)

✓ Batch 13 completed
Processing batch 14/14 (157 tickers)



4 Failed downloads:
['098070.KQ', '452450.KQ']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
['498390.KQ']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-01-11 00:00:00 -> 2025-01-17 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1736521200, endDate = 1737039600")')
['060560.KQ']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-01-11 00:00:00 -> 2025-01-17 00:00:00)')


✓ Batch 14 completed
Combining data for 2703 tickers...
Final data shape: (4, 16218)
Columns: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
date_range = 2일 때 계산---------------------------
Ticker
095570.KS     27.864380
006840.KS     36.015936
027410.KS     31.905554
282330.KS    -14.509601
138930.KS    -50.024310
001460.KS     29.128440
001465.KS    714.285714
001040.KS    -13.563806
079160.KS     10.636193
00104K.KS      5.807463
Name: 2025-01-16 00:00:00, dtype: float64
조건을 만족하는 ticker들 수: 428
최종 선별된 종목 수: 59


{'460940.KQ': '피앤에스미캐닉스',
 '020400.KQ': '대동금속',
 '229000.KQ': '젠큐릭스',
 '298380.KQ': '에이비엘바이오',
 '219420.KQ': '링크제니시스',
 '443670.KQ': '에스피소프트',
 '009460.KS': '한창제지',
 '458870.KQ': '씨어스테크놀로지',
 '073540.KQ': '에프알텍',
 '411080.KQ': '샌즈랩',
 '099220.KQ': 'SDN',
 '046940.KQ': '우원개발',
 '065680.KQ': '우주일렉트로',
 '033100.KQ': '제룡전기',
 '290560.KQ': '신시웨이',
 '430690.KQ': '한싹',
 '476080.KQ': 'M83',
 '101360.KQ': '에코앤드림',
 '353810.KQ': '이지바이오',
 '456010.KQ': '아이씨티케이',
 '160190.KQ': '하이젠알앤엠',
 '000490.KS': '대동',
 '066310.KQ': '큐에스아이',
 '009830.KS': '한화솔루션',
 '456040.KS': 'OCI',
 '475400.KQ': '씨메스',
 '000155.KS': '두산우',
 '009835.KS': '한화솔루션우',
 '299030.KQ': '하나기술',
 '257720.KQ': '실리콘투',
 '315640.KQ': '딥노이드',
 '006260.KS': 'LS',
 '088390.KQ': '이녹스',
 '475960.KQ': '토모큐브',
 '192820.KS': '코스맥스',
 '108490.KQ': '로보티즈',
 '434480.KQ': '모니터랩',
 '053290.KQ': 'NE능률',
 '357550.KQ': '석경에이티',
 '010060.KS': 'OCI홀딩스',
 '110990.KQ': '디아이티',
 '384470.KQ': '코어라인소프트',
 '192250.KQ': '케이사인',
 '000250.KQ': '삼천당제약',
 '041930.KQ